In [17]:
import pandas as pd
import csv


In [18]:
df = pd.read_csv("03_output/df_SHI_with_LC_LU_reduced.csv")

df.head()
print(len(df))

4538


In [19]:
print("Landcover:", df["Landcover"].unique())
print("Landuse:", df["Landuse"].unique())

Landcover: <StringArray>
['Cropland', 'Woodland', 'Grassland', 'Shrubland', 'Bareland']
Length: 5, dtype: str
Landuse: <StringArray>
['Agriculture (excluding fallow land and kitchen gardens)',
                                                'Forestry',
                                             'Fallow land',
               'Semi-natural and natural areas not in use']
Length: 4, dtype: str


In [21]:
# set a threshold limit, combinations between LC and LU that occur equal or less can be deleted
threshold = 29 #29 in Bericht

# count and print these rare combinations
combo_counts = (
    df.groupby(["Landcover", "Landuse"])
      .size()
      .reset_index(name="count")
)

rare = combo_counts[combo_counts["count"] < threshold]
total_rare = rare["count"].sum()

print(f"Rare combinations of Landcover/Landuse with <= {threshold} counts")
print("-" * 71)
print(rare)
print("-" * 71)
print(f"Sum: {total_rare} to be dropped")
print("-" * 71)


# compute frequency of each Landcover/Landuse combination and keep only rows above threshold
mask = df.groupby(["Landcover", "Landuse"])["Landcover"].transform("size") > threshold
df_clean = df[mask].copy()

# check the differences in dataset size
print(f"{'Dataset size before:':<35} {len(df):>6}")
print(f"{'Dataset size after cleaning:':<35} {len(df_clean):>6}")
print(f"{'Rows dropped:':<35} {total_rare:>6}")
print("-" * 71)

print(len(df_clean))

Rare combinations of Landcover/Landuse with <= 29 counts
-----------------------------------------------------------------------
    Landcover                                            Landuse  count
2    Bareland                                           Forestry      2
3    Bareland          Semi-natural and natural areas not in use      2
5    Cropland                                           Forestry      1
8   Grassland                                           Forestry     21
11  Shrubland                                        Fallow land      2
12  Shrubland                                           Forestry     10
14   Woodland  Agriculture (excluding fallow land and kitchen...     18
-----------------------------------------------------------------------
Sum: 56 to be dropped
-----------------------------------------------------------------------
Dataset size before:                  4538
Dataset size after cleaning:          4482
Rows dropped:                           56


In [22]:
# save the df in output folder
df_clean.to_csv("03_output/df_SHI_with_LC_LU_more_reduced.csv", index=False)